In [ ]:
from transformers import DistilBertTokenizerFast
from datasets import load_dataset
import evaluate
class BaseUtilsCola:
    def __init__(self,random_seed=42,prompt_tokens=0):
        self.prompt_tokens = prompt_tokens
        self.tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')
        self.dataset = load_dataset("glue", "cola")
        self.random_seed = random_seed
        self.train_data = self.dataset['train']
        self.val_data = self.dataset['validation']
        self.test_data = self.dataset['test']
        
    def tokenize_function(self,examples):
        '''
        Tokenize the text column of the input examples
        Cuts the text if it is longer than the maximum length of the model
        The responsibilty of the padding is left to the dataloader
        '''
        return self.tokenizer(examples['sentence'], truncation=True,max_length = 512 - self.prompt_tokens)
    
    def get_tokenized_datasets(self):
        '''
        Tokenize the training, validation and test data
        '''
        tokenized_train_data = self.train_data.map(self.tokenize_function, batched=True)
        tokenized_val_data = self.val_data.map(self.tokenize_function, batched=True)
        tokenized_test_data = self.test_data.map(self.tokenize_function, batched=True)
        
        tokenized_train_data = tokenized_train_data.remove_columns(['sentence'])
        tokenized_val_data = tokenized_val_data.remove_columns(['sentence'])
        tokenized_test_data = tokenized_test_data.remove_columns(['sentence'])
        
        return tokenized_train_data, tokenized_val_data, tokenized_test_data
    
    @classmethod
    def compute_metric_mcc(cls, eval_pred):
        '''
        Computes the Matthews correlation coefficient (MCC) for the CoLA dataset.
        '''
        matthews_corr = evaluate.load("matthews_correlation")
        logits, labels = eval_pred
        predictions = logits.argmax(axis=-1)
        return matthews_corr.compute(predictions=predictions, references=labels)
    
    @classmethod
    def write_time(cls,start_time,end_time,batch_size,epochs):
        '''
        It writes the time taken to train the model
        '''
        with open('time.txt','a+') as f:
            f.write(f'Batch size {batch_size}:epochs={epochs}:{int(end_time-start_time)} seconds\n')
            
    def save_predictions(self, predictions,name):
        import pandas as pd

        df = pd.DataFrame({'Label':predictions,"Id":list(range(1,len(predictions)+1))})
        df.to_csv(name+'.csv', index=False)

In [ ]:
import torch
from time import time
import pandas as pd
from transformers import set_seed

In [ ]:
set_seed(42)
torch.manual_seed(42)
batch_size = 32
epochs = 3
lr=5e-5
shuffle = True
clean_text = True
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
selection_criterion = 'eval_loss' # Choose between 'accuracy' and 'eval_loss'
prompt_tokens = 30

In [ ]:
base_utils_obj = BaseUtilsCola(prompt_tokens=prompt_tokens)
tokenized_train, tokenized_val, tokenized_test = base_utils_obj.get_tokenized_datasets()
results_path = './results/batch_size_{}_epochs_{}_lr_{}_selection_criterion_{}'.format(batch_size, epochs, lr, selection_criterion)
log_path = './logs/batch_size_{}_epochs_{}_lr_{}_selection_criterion_{}'.format(batch_size, epochs, lr, selection_criterion)

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

model_name = 'distilbert-base-uncased'
    
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
model.to(device)

In [ ]:
from peft import get_peft_model, PromptTuningConfig, TaskType

# Step 1: Create config (without prompt_projection)
peft_config = PromptTuningConfig(
    task_type=TaskType.SEQ_CLS,    # Sequence Classification
    inference_mode=False,          # Training mode
    num_virtual_tokens=prompt_tokens,         # Number of prompt tokens
    token_dim=768,
    num_layers=6,    # DistilBERT hidden size
    num_attention_heads=12,         # DistilBERT number of attention heads
)
peft_config.modules_to_save = None
# Step 2: Wrap your model
model = get_peft_model(model, peft_config)

# Step 3: Print trainable parameters
model.print_trainable_parameters()

In [ ]:
from transformers import TrainingArguments, Trainer

In [ ]:
training_args = TrainingArguments(
    output_dir=results_path,
    num_train_epochs=epochs,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=256,
    learning_rate=lr,
    logging_dir=log_path,
    logging_steps=50,
    eval_strategy='steps',
    save_steps=50,
    eval_steps=50,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model=selection_criterion,
    greater_is_better=False,
    report_to='tensorboard',
    seed=42,
    run_name='Batch size: {}, Epochs: {}, LR: {}, selection_criterion: {}'.format(batch_size, epochs, lr, selection_criterion),
    lr_scheduler_type='constant',
    warmup_steps=0,
    fp16=False,
    label_names=["label"]
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    processing_class=base_utils_obj.tokenizer,
    compute_metrics=base_utils_obj.compute_metric_mcc,
)

In [ ]:
start_time = time()

In [ ]:
trainer.train()

In [ ]:
# Save the best model
'''
Keep a local copy of the best model
'''
best_model_path = "./best_model_{batch_size}_{epochs}_{lr}_selection_criterion_{selection_criterion}".format(batch_size=batch_size, epochs=epochs, lr=lr, selection_criterion=selection_criterion)
trainer.model.save_pretrained(best_model_path)

In [ ]:
base_utils_obj.write_time(start_time,time(),batch_size,epochs)

In [ ]:
df1 = pd.read_csv("./cola_in_domain_test.tsv", sep="\t")
df2 = pd.read_csv("./cola_out_of_domain_test.tsv", sep="\t")
df1 = df1.rename(columns={"Sentence": "sentence"})
df2 = df2.rename(columns={"Sentence": "sentence"})

In [ ]:
from datasets import Dataset
test_in_domain = Dataset.from_pandas(df1[['sentence']])
test_out_of_domain = Dataset.from_pandas(df2[['sentence']])
test_in_domain = test_in_domain.map(base_utils_obj.tokenize_function, batched=True)
test_out_of_domain = test_out_of_domain.map(base_utils_obj.tokenize_function, batched=True)

In [ ]:
in_domain_predictions = trainer.predict(test_in_domain)
out_of_domain_predictions = trainer.predict(test_out_of_domain)
in_domain_pred_labels = in_domain_predictions.predictions.argmax(-1)
out_of_domain_pred_labels = out_of_domain_predictions.predictions.argmax(-1)

In [ ]:
base_utils_obj.save_predictions(in_domain_pred_labels,name='in_domain_predictions')
base_utils_obj.save_predictions(out_of_domain_pred_labels,name='out_of_domain_predictions')